## We aim in this notebook to identify duplicates in a CSV containing information about restaurants

In [1]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import nltk
import time
import numpy as np

In [4]:
df_restaurants = pd.read_csv("./restaurants.csv")

In [6]:
df_restaurants.head(10)

,name,address,city,cuisine,unique_id
0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,21 club,21 w. 52nd st.,new york,american,23
3,2223,2223 market st.,san francisco,american,453
4,9 jones street,9 jones st.,new york,american,173
5,abbey,163 ponce de leon ave.,atlanta,international,379
6,abruzzi,2355 peachtree rd. peachtree battle shopping...,atlanta,italian,74
7,acquarello,1722 sacramento st.,san francisco,italian,454
8,adrianos ristorante,2930 beverly glen circle,los angeles,italian,112
9,adrienne,700 5th ave. at 55th st.,new york,french,174


The dataframe contains duplicates records that represent to the same 'real-world' restanrants. The column 'unique_id' was added for this purpose. Two records that are associated with the same attribute value for unique_id represents the same restaurant.

In [9]:
df_restaurants[df_restaurants.unique_id == '23']

,name,address,city,cuisine,unique_id
2,21 club,21 w. 52nd st.,new york,american,23
753,21 club,21 w. 52nd st.,new york city,american (new),23


In the above example, the two records share the same value for attributes 'name' and 'address'. However, they have slightly different values for the columns 'city' and 'cuisine'

In [10]:
df_restaurants[df_restaurants.unique_id == '22']

,name,address,city,cuisine,unique_id
744,yujean kangs gourmet chinese cuisine,67 n. raymond ave.,los angeles,asian,22
864,yujean kangs,67 n. raymond ave.,pasadena,chinese,22


In the above example, on the other hand, the two records are associated with different names, cities and cuisiones.

This file represents a simple example of datasets, on which we can experiment with th etechniques presented in the course to try identify duplicates, without using (that is relying on the values of) the column "unique_id".

In [11]:
# We start by adding a new column to identify the records (lines) in our dataframe
df_restaurants.insert(0,'record_ID', range(0, len(df_restaurants)))

In [8]:
df_restaurants.head(5)

,record_ID,name,address,city,cuisine,unique_id
0,0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,2,21 club,21 w. 52nd st.,new york,american,23
3,3,2223,2223 market st.,san francisco,american,453
4,4,9 jones street,9 jones st.,new york,american,173


In [12]:
df_restaurants

,record_ID,name,address,city,cuisine,unique_id
0,0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,2,21 club,21 w. 52nd st.,new york,american,23
3,3,2223,2223 market st.,san francisco,american,453
4,4,9 jones street,9 jones st.,new york,american,173
...,...,...,...,...,...,...
860,860,union square cafe,21 e. 16th st.,new york city,american (new),65
861,861,valentino,3115 pico blvd.,santa monica,italian,21
862,862,veni vidi vici,41 14th st.,atlanta,italian,93
863,863,virgils real bbq,152 w. 44th st.,new york city,bbq,66


# Exhaustive comparisons: every record is compared with every other record

We start by applying an exhaustive strategy whereby every record in the CSV file, is compared with every other record. 

The code below does this for us. In doing so, it uses the following rule:

For two records to match, i.e. refer to the same restaurant in the real world:
* The edit distance between the attribute name values of the two records needs to be smaller or equal to 2, and 
* they need to have the same value for the cuisine attribute.


In [13]:
num_records = len(df_restaurants)
matches = []
number_of_matches = 0
start = time.process_time()
for i in range(0,num_records):
    for j in range(i+1,num_records):
        name_score = nltk.edit_distance(df_restaurants.iloc[i,1], df_restaurants.iloc[j,1])
        
        # Rule for matching: Distance between names is smaller or equal to 3 and the cuisine is the same 
        if (name_score <= 2) and (df_restaurants.iloc[i,4] == df_restaurants.iloc[j,4]):
            number_of_matches = number_of_matches +1 
            matches.append((df_restaurants.iloc[i,0],df_restaurants.iloc[j,0]))

end = time.process_time()

print("Number of matches: {}".format(number_of_matches))
print("Processing time: {}".format(end - start))

Number of matches: 32
Processing time: 152.265625


In [14]:
# Display results
for match in matches:
    print("The following records {} and {} match".format(match[0],match[1]))
    print("The restaurants with the following names {} and {} match.".format(df_restaurants.iloc[match[0],1],df_restaurants.iloc[match[1],1]))
    print("The restaurants with the following addresses {} and {} match.".format(df_restaurants.iloc[match[0],2],df_restaurants.iloc[match[1],2]))
    print("\n")

The following records 6 and 754 match
The restaurants with the following names abruzzi and abruzzi match.
The restaurants with the following addresses  2355 peachtree rd.  peachtree battle shopping center and  2355 peachtree rd. ne match.


The following records 55 and 56 match
The restaurants with the following names bertolinis and bertolinis match.
The restaurants with the following addresses  3500 peachtree rd.  phipps plaza and  3570 las vegas blvd. s match.


The following records 87 and 88 match
The restaurants with the following names bruno and brunos match.
The restaurants with the following addresses  240 e. 58th st. and  3838 centinela ave. match.


The following records 141 and 773 match
The restaurants with the following names carmines and carmines match.
The restaurants with the following addresses  2450 broadway  between 90th and 91st sts. and  2450 broadway match.


The following records 153 and 154 match
The restaurants with the following names cha cha cha and cha cha c

Note that the rule applied in the above code is not great. You may want to try other kind of distances, other thresholds, and other rules to identify matches.

<span style='color:blue'>
Try to change the code above, use different rules and change the distances used for matching the attributes values. In essence, a good entity resolution program needs to minimize the overall execution time without compromising the quality of the results. Before doing so, we show below how can we assess the results obtained by comparing them to the ground truth results.    
</span>

# Assessing the quality of the results

To do so, we first need to compute the ground truth (that is the list of correct matches) considering the attribute unique_id.

In [16]:
ground_truth_matches = pd.read_csv("./restaurants.csv")
ground_truth_matches.head(5)

,name,address,city,cuisine,unique_id
0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,21 club,21 w. 52nd st.,new york,american,23
3,2223,2223 market st.,san francisco,american,453
4,9 jones street,9 jones st.,new york,american,173


In [17]:
ground_truth_matches.insert(0, 'record_ID', range(0, len(ground_truth_matches)))

In [18]:
ground_truth_matches.head(5)

,record_ID,name,address,city,cuisine,unique_id
0,0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,2,21 club,21 w. 52nd st.,new york,american,23
3,3,2223,2223 market st.,san francisco,american,453
4,4,9 jones street,9 jones st.,new york,american,173


In [21]:
ground_truth_matches = pd.merge(ground_truth_matches,
                                ground_truth_matches,
                                on = 'unique_id')

In [22]:
ground_truth_matches.head(5)

,record_ID_x,name_x,address_x,city_x,cuisine_x,unique_id,record_ID_y,name_y,address_y,city_y,cuisine_y
0,0,103 west,103 w. paces ferry rd.,atlanta,continental,675,0,103 west,103 w. paces ferry rd.,atlanta,continental
1,1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172,1,20 mott,20 mott st. between bowery and pell st.,new york,asian
2,2,21 club,21 w. 52nd st.,new york,american,23,2,21 club,21 w. 52nd st.,new york,american
3,2,21 club,21 w. 52nd st.,new york,american,23,753,21 club,21 w. 52nd st.,new york city,american (new)
4,753,21 club,21 w. 52nd st.,new york city,american (new),23,2,21 club,21 w. 52nd st.,new york,american


In [23]:
ground_truth_matches = ground_truth_matches.query('record_ID_x < record_ID_y')

In [24]:
ground_truth_matches.head(5)

,record_ID_x,name_x,address_x,city_x,cuisine_x,unique_id,record_ID_y,name_y,address_y,city_y,cuisine_y
3,2,21 club,21 w. 52nd st.,new york,american,23,753,21 club,21 w. 52nd st.,new york city,american (new)
10,6,abruzzi,2355 peachtree rd. peachtree battle shopping...,atlanta,italian,74,754,abruzzi,2355 peachtree rd. ne,atlanta,italian
20,13,alain rondelli,126 clement st.,san francisco,french,94,755,alain rondelli,126 clement st.,san francisco,french (new)
36,26,aquavit,13 w. 54th st.,new york,continental,24,756,aquavit,13 w. 54th st.,new york city,scandinavian
40,27,aqua,252 california st.,san francisco,seafood,95,757,aqua,252 california st.,san francisco,american (new)


In [25]:
ground_truth_matches = ground_truth_matches[['record_ID_x','record_ID_y']]

In [26]:
print(ground_truth_matches)

      record_ID_x  record_ID_y
3               2          753
10              6          754
20             13          755
36             26          756
40             27          757
...           ...          ...
1030          708          860
1034          709          861
1041          713          862
1053          722          863
1078          744          864

[112 rows x 2 columns]


In [27]:
matches_df = pd.DataFrame(matches)
matches_df.columns= ['record_ID_x','record_ID_y']

In [28]:
matches_df.head()

,record_ID_x,record_ID_y
0,6,754
1,55,56
2,87,88
3,141,773
4,153,154


In [29]:
diff_df = pd.merge(ground_truth_matches, matches_df, how='outer', indicator='Exist')

In [31]:
diff_df.head()

,record_ID_x,record_ID_y,Exist
0,2,753,left_only
1,6,754,both
2,13,755,left_only
3,26,756,left_only
4,27,757,left_only


In [32]:
true_positives = diff_df[diff_df.Exist=='both']
false_positives = diff_df[diff_df.Exist=='right_only']
false_negatives = diff_df[diff_df.Exist=='left_only']

In [33]:
true_positives.head()

,record_ID_x,record_ID_y,Exist
1,6,754,both
20,141,773,both
27,170,780,both
33,233,785,both
41,278,793,both


In [34]:
false_positives.head()

,record_ID_x,record_ID_y,Exist
112,55,56,right_only
113,87,88,right_only
114,153,154,right_only
115,161,164,right_only
116,169,180,right_only


In [35]:
false_negatives.head()

,record_ID_x,record_ID_y,Exist
0,2,753,left_only
2,13,755,left_only
3,26,756,left_only
4,27,757,left_only
5,30,758,left_only


In [36]:
df_restaurants

,record_ID,name,address,city,cuisine,unique_id
0,0,103 west,103 w. paces ferry rd.,atlanta,continental,675
1,1,20 mott,20 mott st. between bowery and pell st.,new york,asian,172
2,2,21 club,21 w. 52nd st.,new york,american,23
3,3,2223,2223 market st.,san francisco,american,453
4,4,9 jones street,9 jones st.,new york,american,173
...,...,...,...,...,...,...
860,860,union square cafe,21 e. 16th st.,new york city,american (new),65
861,861,valentino,3115 pico blvd.,santa monica,italian,21
862,862,veni vidi vici,41 14th st.,atlanta,italian,93
863,863,virgils real bbq,152 w. 44th st.,new york city,bbq,66


In [38]:
#Example of a true positive
df_restaurants[df_restaurants.record_ID.isin([6,754])]

,record_ID,name,address,city,cuisine,unique_id
6,6,abruzzi,2355 peachtree rd. peachtree battle shopping...,atlanta,italian,74
754,754,abruzzi,2355 peachtree rd. ne,atlanta,italian,74


In [39]:
#Example of a false positive
df_restaurants[df_restaurants.record_ID.isin([55,56])]

,record_ID,name,address,city,cuisine,unique_id
55,55,bertolinis,3500 peachtree rd. phipps plaza,atlanta,italian,385
56,56,bertolinis,3570 las vegas blvd. s,las vegas,italian,427


In [40]:
#Example of a false negative
df_restaurants[df_restaurants.record_ID.isin([2,753])]

,record_ID,name,address,city,cuisine,unique_id
2,2,21 club,21 w. 52nd st.,new york,american,23
753,753,21 club,21 w. 52nd st.,new york city,american (new),23


In [41]:
precision = len(true_positives)/(len(true_positives)+ len(false_positives))
print(precision)

0.5625


Note that if you are using pyton 2.7 (instead of Python 3), you would need to convert integers to float prior to performing the division

In [42]:
recall = len(true_positives)/(len(true_positives)+ len(false_negatives))
print(recall)

0.16071428571428573


In [43]:
f_measure = 2*(precision*recall)/(precision+recall)
print(f_measure)

0.25000000000000006


<span style='color:blue'>
The above results is not a good one, this is due to the rule chosen when decising if two records match or not. 
Try to modify the rules, by first examining the dataset, to see if you can improve the precision and recall.
</span>

# Windowing (SNM) method

In [48]:
def snm(df_restaurants, window_size, sort_attributes):
    """
    Sorted Neighborhood Method (SNM) for record linkage.
    
    Parameters:
    - df_restaurants: DataFrame containing the restaurant records
    - window_size: Size of the window to slide over sorted tuples
    - sort_attributes: List of column names to sort the records
    
    Returns:
    - matches: List of unique matched record pairs
    """
    # Step 1: Sort the DataFrame based on the provided attributes
    df_sorted = df_restaurants.sort_values(by=sort_attributes).reset_index(drop=True)
    
    num_records = len(df_sorted)
    matches = []
    number_of_matches = 0
    
    # Start measuring time
    start = time.process_time()

    # Step 2: Sliding window
    for i in range(0, num_records - window_size + 1):
        # Define the window range
        window_records = df_sorted.iloc[i:i+window_size]

        # Step 3: Compare each pair of records within the window
        for j in range(0, len(window_records)):
            for k in range(j+1, len(window_records)):
                # Extract the two records
                record_1 = window_records.iloc[j]
                record_2 = window_records.iloc[k]

                # Step 4: Apply the same matching logic as before
                name_score = nltk.edit_distance(record_1['name'], record_2['name'])
                
                # If the distance is small, cuisine matches, and it's not a self-match
                if (name_score <= 2) and (record_1['cuisine'] == record_2['cuisine']) :
                    number_of_matches += 1
                    matches.append((record_1['unique_id'], record_2['unique_id']))

    # Remove duplicates by converting to a set and back to a list
    matches = list(set(matches))

    # End measuring time
    end = time.process_time()

    print(f"Number of unique matches: {len(matches)}")
    print(f"Processing time: {end - start}")
    
    return matches

# Re-run the updated SNM method with print statements to track self-matches
df_restaurants = pd.read_csv("./restaurants.csv")
df_restaurants.insert(0, 'record_ID', range(0, len(df_restaurants)))
window_size = 3
sort_attributes = ['name', 'address']

matches_new_dataset  = snm(df_restaurants, window_size, sort_attributes)
matches_new_dataset 


Number of unique matches: 27
Processing time: 0.6875


[('16', '16'),
 ('93', '93'),
 ('57', '57'),
 ('33', '33'),
 ('53', '53'),
 ('74', '74'),
 ('8', '8'),
 ('85', '85'),
 ('286', '144'),
 ('54', '54'),
 ('48', '48'),
 ('87', '87'),
 ('310', '148'),
 ('334', '159'),
 ('195', '544'),
 ('28', '28'),
 ('385', '427'),
 ('45', '45'),
 ('261', '137'),
 ('6', '6'),
 ('111', '111'),
 ('699', '564'),
 ('124', '725'),
 ('21', '21'),
 ('72', '671'),
 ('50', '50'),
 ('13', '13')]

<span style='color:blue'>
In this section, you will have to implement the SNM method. It will take as parameters the size of the window, and the attributes based on which the tuples will be ordered.
</span>

# Blocking method

<span style='color:blue'>
Now that we have seen how the exhaustive and windowing methods work, can you try to examine how the blocking method can be implemented. In oparticular, can you spacify how the blocks can be formed; To do so, examine the restaurant dataset.
</span>

In [50]:
# Re-define the blocking method function before running it
def blocking_method(df_restaurants, block_keys):
    """
    Blocking method for record linkage.
    
    Parameters:
    - df_restaurants: DataFrame containing the restaurant records
    - block_keys: List of column names to use for blocking (e.g., ['city', 'name'])
    
    Returns:
    - matches: List of matched record pairs within blocks
    """
    # Step 1: Create blocks based on the provided block keys
    df_restaurants['block'] = df_restaurants[block_keys].apply(lambda x: '_'.join(x.astype(str)), axis=1)
    
    # Group the records by the block
    grouped_blocks = df_restaurants.groupby('block')
    
    matches = []
    
    # Step 2: Compare records within each block
    for block, group in grouped_blocks:
        num_records = len(group)
        
        # Compare each pair within the block
        for i in range(num_records):
            for j in range(i+1, num_records):
                record_1 = group.iloc[i]
                record_2 = group.iloc[j]
                
                # Compare restaurant names using edit distance
                name_score = nltk.edit_distance(record_1['name'], record_2['name'])
                
                # If names are similar and cuisine is the same, add to matches
                if name_score <= 2 and record_1['cuisine'] == record_2['cuisine']:
                    matches.append((record_1['unique_id'], record_2['unique_id']))
    
    return matches

# Define the blocking keys: city and first letter of name
df_restaurants = pd.read_csv("./restaurants.csv")
df_restaurants['name_first_letter'] = df_restaurants['name'].str[0]  # Extract first letter of the name
block_keys = ['city', 'name_first_letter']

# Run the blocking method on the new dataset
matches_blocking_method = blocking_method(df_restaurants, block_keys)

# Display the matches found using the blocking method
matches_blocking_method

# Display results
for match in matches_blocking_method:
    print("The following records {} and {} match".format(match[0],match[1]))
    print("The restaurants with the following names {} and {} match.".format(df_restaurants.iloc[match[0],1],df_restaurants.iloc[match[1],1]))
    print("The restaurants with the following addresses {} and {} match.".format(df_restaurants.iloc[match[0],2],df_restaurants.iloc[match[1],2]))
    print("\n")



The following records 74 and 74 match


ValueError: Location based indexing can only have [integer, integer slice (START point is INCLUDED, END point is EXCLUDED), listlike of integers, boolean array] types